In [ ]:
import xarray as xr
import pandas as pd
import glob
import re
import re
from datetime import datetime
from pathlib import Path

from pace_pic_validation.viz.RGB_functions import rgb_plot_with_stations




In [ ]:
path_merge_ds = '/Volumes/Extreme SSD/Bloom 202506/8D_20250502_20250701'
path_all_pic  = '/Volumes/Extreme SSD/PhD backup/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/PIC_discrete/all_PIC.csv'


# seperating stations of all_pic

df = pd.read_csv(path_all_pic)
df = df.sort_values('datetime')
print(df.columns)

groups = {k: v.copy() for k, v in df.groupby("Campaign")}
print()


SS_campaigns= {k: v for k, v in groups.items() if k.startswith("SS")}



print(SS_campaigns.keys())

# merged datasets handling
paths_merged_ds = glob.glob(path_merge_ds + '/*.nc')

paths_sorted = sorted(paths_merged_ds)

for p in paths_sorted:
    print(p)
    p=p.replace("'", "")
    print(p)


df = SS_campaigns['SS20260601']

#df = df.where(df['Station'].str.endswith(('05', '06', '07', '08', '09', '10', '11')))
df = df.dropna(axis = 'rows')
df = df.reset_index()

ds = xr.open_dataset(paths_merged_ds[0])
print(ds['wavelength'])

In [ ]:


def extract_start_date_from_nc(path: str) -> str | None:
    """
    Extract the start date from a filename like:
    2025-06-02_2025-06-06_latmin_48.0_..._['CLDICE'].nc
    Returns date as 'YYYYMMDD' or None if not found.
    """
    fname = Path(path).name
    m = re.match(r"(\d{4}-\d{2}-\d{2})", fname)
    if not m:
        return None
    date_str = m.group(1)  # "2025-06-02"
    # Convert to YYYYMMDD
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    return dt.strftime("%Y%m%d")

from pathlib import Path

# Map from date string (YYYYMMDD) -> nc file path (or loaded dataset)
date_to_nc = {}
for f in paths_sorted:
    d = extract_start_date_from_nc(f)
    if d is not None:
        date_to_nc[d] = f  # or xr.open_dataset(f) if you want to load now

# Now match with ss_dict keys
matched = {}
for key, ds in SS_campaigns.items():
    # key is like "SS20250602"
    date_part = key[2:]  # "20250602"
    if date_part in date_to_nc:
        matched[key] = (ds, date_to_nc[date_part])
    else:
        matched[key] = (ds, None)  # or handle missing case



In [ ]:

path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC\ Product\ Validation\ for\ PACE/Data/merged_datasets/2025-06-01_2025-06-22_latmin_50.0_latmax_65.0_lonmin_-35.0_lonmax_1.0_\[\'CLDICE\'\].nc'
df = SS_campaigns['SS20250602']
df = df.where(df['Station'].str.endswith(('05', '06', '07', '08', '09', '10', '11')))

for path in paths_sorted:

    import re
    match = re.search(r"\d{4}-\d{2}-\d{2}_\d{4}-\d{2}-\d{2}", path)

    if match:
        title = match.group(0)
        print(title)

    ds = xr.open_dataset(path)

    rgb_plot_with_stations(
        ds=ds,
        df=df,
        show_station_labels=True,
        show_stations=True,
        gridlines=True,
        save=True,
        title=title,
        extent = [-1, 7, 54, 58],
        save_dir='/Volumes/Extreme SSD/Bloom 202506/8D_rgbs',
        dpi = 1000
        )


In [ ]:



for ss_key, (ss_df, nc_path) in matched.items():
    if nc_path is None:
        continue
    ds = xr.open_dataset(nc_path)

    rgb_plot_with_stations(
    ds=ds,
    df=ss_df,
    show_station_labels=True,
    show_stations=True,
    gridlines=True,
    save=False,
    title=ss_key,
    extent = [1, 4.5, 54.5, 57],
    save_dir='/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Figures/Ocean Optics',
    )

    
